# Importar pacotes

In [1]:
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

# Métodos utilitários

In [2]:
from utils.common import import_dataframe
from utils.common import make_lags, make_leads
from utils.common import calculate_metrics
from utils.common import HybridRecursive, HybridBase
from utils.common import ModeloPrevisaoVolume

# Importação dos dados

In [3]:
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df_volume = df[["Data", nome_coluna_volume]].copy()
df_vazao_natural = df[["Data", nome_coluna_vazao_natural]].copy()
df_vazao_jusante = df[["Data", nome_coluna_vazao_jusante]].copy()

df_volume = df_volume.set_index("Data")
df_vazao_natural = df_vazao_natural.set_index("Data")
df_vazao_jusante = df_vazao_jusante.set_index("Data")

df_volume_series = (
  df_volume
    .groupby('Data').mean()
    .squeeze()
)
df_vazao_natural_series = (
  df_vazao_natural
    .groupby('Data').mean()
    .squeeze()
)

df_vazao_jusante_series = (
  df_vazao_jusante
    .groupby('Data').mean()
    .squeeze()
)

# Modelagem - Vazão Natural

In [8]:
y_vn = df_vazao_natural_series[df_vazao_natural_series.index >= "2018-01-01"].copy()

fourier = CalendarFourier(freq="YE", order=2)
y_vn = y_vn.asfreq("D")
dp = DeterministicProcess(
  index=y_vn.index,
  constant=False,
  order=1,
  seasonal=False,
  additional_terms=[fourier],
  drop=True,
)
X_full_vn = dp.in_sample()

VALIDATION_SIZE = 1*90

X_vn_train_rec, X_vn_valid_rec, y_vn_train_rec, y_vn_valid_rec = train_test_split(X_full_vn, y_vn, test_size=VALIDATION_SIZE, shuffle=False)

## Treino

In [6]:
model = HybridRecursive(Lasso(), KNeighborsRegressor(), lags=2)
model.fit(X_vn_train_rec, y_vn_train_rec)
model.calculate_lags(X_vn_valid_rec)

X_vn_train_rec_drop = X_vn_train_rec.drop(pd.Timestamp("2018-01-01"))
y_fit = model.predict(X_vn_train_rec_drop)
y_pred = model.predict(X_vn_valid_rec)

# Modelagem - Vazão Jusante

In [9]:
y_vj = df_vazao_jusante_series[df_vazao_jusante_series.index >= "2018-01-01"].copy()

fourier = CalendarFourier(freq="YE", order=2)
y_vj = y_vj.asfreq("D")
dp = DeterministicProcess(
  index=y_vj.index,
  constant=False,
  order=1,
  seasonal=False,
  additional_terms=[fourier],
  drop=True,
)
X_full_vj = dp.in_sample()

VALIDATION_SIZE = 1*90

X_vj_train_rec, X_vj_valid_rec, y_vj_train_rec, y_vj_valid_rec = train_test_split(X_full_vj, y_vj, test_size=VALIDATION_SIZE, shuffle=False)

## Treino

In [10]:
model = HybridRecursive(LinearRegression(), RandomForestRegressor(), lags=2)
model.fit(X_vj_train_rec, y_vj_train_rec)
model.calculate_lags(X_vj_valid_rec)

X_vj_train_rec_drop = X_vj_train_rec.drop(pd.Timestamp("2018-01-01"))
y_fit = model.predict(X_vj_train_rec_drop)
y_pred = model.predict(X_vj_valid_rec)

c:\Users\marce\Desktop\Pos Graduacoes\UFSCAR - MachineLearningInProduction\AtividadesEntregas\4_TCC\codigo\tcc\.venv\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


# Modelagem - Vazão Volume

In [11]:
def get_train_test_split_recursive():
  y_vol = df_volume_series[df_volume_series.index >= "2018-01-01"].copy()
  X_lags = make_lags(y_vol.squeeze(), 1)
  X_Qin_leads = make_leads(y_vn.squeeze(), 1, name="Qin")
  X_Qout_leads = make_leads(y_vj.squeeze(), 1, name="Qout")
  X_full_vol = pd.concat([X_lags, X_Qin_leads, X_Qout_leads], axis=1).dropna()

  y_vol, X_full_vol = y_vol.align(X_full_vol, join='inner', axis=0)

  X_vol_train_rec, X_vol_valid_rec, y_vol_train_rec, y_vol_valid_rec = train_test_split(X_full_vol, y_vol, test_size=VALIDATION_SIZE, shuffle=False)
  return y_vol, X_full_vol, X_vol_train_rec, X_vol_valid_rec, y_vol_train_rec, y_vol_valid_rec

y_vol_rec, X_full_vol, X_vol_train_rec, X_vol_valid_rec, y_vol_train_rec, y_vol_valid_rec = get_train_test_split_recursive()

## Treino

In [12]:
model = ModeloPrevisaoVolume(LinearRegression(fit_intercept=False))

model.fit_vazao_natural(X_vn_train_rec, y_vn_train_rec)
model.fit_vazao_jusante(X_vj_train_rec, y_vj_train_rec)
model.fit(X_vol_train_rec, y_vol_train_rec)
model.calculate_lags(X_vn_valid_rec)

X_vn_train_rec_drop = X_vn_train_rec.drop(pd.Timestamp("2018-01-01"))
y_fit = model.predict(X_vn_train_rec_drop)
y_pred = model.predict(X_vn_valid_rec)

c:\Users\marce\Desktop\Pos Graduacoes\UFSCAR - MachineLearningInProduction\AtividadesEntregas\4_TCC\codigo\tcc\.venv\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


### Testando outros modelos.

In [ ]:
def get_model_name(model):
  return f'{model}'.split('(')[0]

models_for_trend = [LinearRegression(), ElasticNet(), Lasso(), Ridge()]
models_for_cycle = [LinearRegression(), XGBRegressor(), ExtraTreesRegressor(), 
                    RandomForestRegressor(), MLPRegressor(), KNeighborsRegressor()]

for trend_model in models_for_trend:
  for cycle_model in models_for_cycle:
    model_name_trend = get_model_name(trend_model)
    model_name_cycle = get_model_name(cycle_model)
    model_name = model_name_trend + '/' + model_name_cycle
    print("*"*90)
    print("Modelo Tendência:", model_name_trend, "\t\t\t","Modelo Ciclo:", model_name_cycle)

    print("="*5, " Estratégia Recursiva ", "="*5)
    rodar_estrategia_recursiva(trend_model, cycle_model, model_name)
    print("="*5, " Estratégia Direta ", "="*5)
    rodar_estrategia_direta(trend_model, cycle_model, model_name)

    # Dir Rec demora muito para alguns dos modelos.
    # Talvez seja necessário diminuir o horizonte de previsão.
    #print("="*5, " Estratégia Dir Rec ", "="*5)
    #rodar_estrategia_dir_rec(trend_model, cycle_model, model_name)

## Resultados

### Estratégia Recursiva

In [ ]:
result_df = pd.concat(results_rec)

def highlight_val_only(s):
  if 'Validação' not in s.index.get_level_values('Dados'):
    return [''] * len(s)
  mask = s.index.get_level_values('Dados') == 'Validação'

  if s.name == 'R2':
    best_val = s[mask].max()
    worst_val = s[mask].min()
  else:
    best_val = s[mask].min()
    worst_val = s[mask].max()

  return [
    'background-color: #ff3333; color: white; font-weight: bold;' if v == worst_val and m else
    'background-color: #00cc66; color: white; font-weight: bold;' if v == best_val and m else ''
      for v, m in zip(s, mask)
  ]

styled = (
  result_df.style
  .apply(highlight_val_only, subset=['RMSE', 'MSE', 'MAE', 'R2', 'Wasserstein'])
  .format("{:.4f}")
)
display(styled)

### Estratégia Direta

In [ ]:
result_df = pd.concat(results_dir)

def highlight_val_only(s):
  if 'Validação' not in s.index.get_level_values('Dados'):
    return [''] * len(s)
  mask = s.index.get_level_values('Dados') == 'Validação'

  if s.name == 'R2':
    best_val = s[mask].max()
    worst_val = s[mask].min()
  else:
    best_val = s[mask].min()
    worst_val = s[mask].max()

  return [
    'background-color: #ff3333; color: white; font-weight: bold;' if v == worst_val and m else
    'background-color: #00cc66; color: white; font-weight: bold;' if v == best_val and m else ''
      for v, m in zip(s, mask)
  ]

styled = (
  result_df.style
  .apply(highlight_val_only, subset=['RMSE', 'MSE', 'MAE', 'R2', 'Wasserstein'])
  .format("{:.4f}")
)
display(styled)

# Teste: cálculo de Volume a partir das vazões

- $ \frac{dv}{dt} = \alpha \cdot (Q_{in}(t) - Q_{out}(t)) $

- $ V(t) = V(t_0) + \alpha \int_{t_0}^{t} \big[ Q_{in}(\tau) - Q_{out}(\tau) \big] \, d\tau $

In [ ]:
y_vol = df[["Data", "SC_VolumeTotalHm3"]].copy()
X = df[["Data", "SC_VazaoNatural", "SC_VazaoJusante"]].copy()

y_vol = (
  y_vol
    .groupby('Data').mean()
    .squeeze()
)
y = y_vol.diff().fillna(0.0)

X = (
  X
    .groupby('Data').mean()
    .squeeze()
)

model = LinearRegression()
model.fit(X, y)

y_pred = pd.Series(model.predict(X), index=X.index, name=y.to_frame().columns[0])
ax = y.plot(**plot_params, alpha=0.5, title="Volume - diferenças", ylabel="diferença (hm³)")
ax = y_pred.plot(ax=ax, linewidth=1, label="Tendência", color='C0')
ax.legend();

y_pred[0] = y_pred[0] + y_vol[0]

In [ ]:
eq_text = print_equation(model)
print(eq_text)

ax = y_vol.plot(**plot_params, alpha=0.5, title="Volume", ylabel="volume (hm³)")
ax = y_pred.cumsum().plot(ax=ax, linewidth=3, label="Tendência", color='C0')
ax.legend();

In [ ]:
model.coef_ = np.array([0.0864, -0.0864])
y_pred = pd.Series(model.predict(X), index=X.index, name=y.to_frame().columns[0])

y_pred = y_pred[1:]
y_pred_diff = y_pred.diff().fillna(0.0)

ax = y.plot(**plot_params, alpha=0.5, title="Volume", ylabel="volume (hm³)")
ax = y_pred_diff.plot(ax=ax, linewidth=1, label="Tendência", color='C0')

ax.legend();